# prometheus_api_client

## Example 1: an instant query, as Dynamo's planner sends one

In [ ]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

from prometheus_api_client import PrometheusConnect

class StandIn(BaseHTTPRequestHandler):  # answers the way Prometheus's HTTP API does
    def log_message(self, *args): pass
    def do_GET(self):
        url = urlparse(self.path)
        query = parse_qs(url.query).get("query", [""])[0]
        result = [{"metric": {"model": "llama"}, "value": [1700000000.0, "42.5"]}] if "rate(" in query else []
        self.send_response(200 if url.path in ("/", "/api/v1/query") else 404); self.end_headers()
        self.wfile.write(json.dumps({"status": "success", "data": {"resultType": "vector", "result": result}}).encode())

server = ThreadingHTTPServer(("127.0.0.1", 0), StandIn)
threading.Thread(target=server.serve_forever, daemon=True).start()
prom = PrometheusConnect(url=f"http://127.0.0.1:{server.server_address[1]}", disable_ssl=True)
assert prom.check_prometheus_connection()
rows = prom.custom_query("sum by (model) (rate(dynamo_frontend_requests_total[1m]))")
assert rows == [{"metric": {"model": "llama"}, "value": [1700000000.0, "42.5"]}], rows
assert prom.custom_query("up") == []
server.shutdown()
print("requests per second for llama:", float(rows[0]["value"][1]))

## Example 2: a range query into a `MetricRangeDataFrame`

In [ ]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse
from datetime import datetime, timedelta, timezone
from prometheus_api_client import MetricRangeDataFrame, PrometheusConnect

class StandIn(BaseHTTPRequestHandler):  # answers the way Prometheus's HTTP API does
    def log_message(self, *args): pass
    def do_GET(self):
        args = {k: v[0] for k, v in parse_qs(urlparse(self.path).query).items()}
        start, end, step = int(args["start"]), int(args["end"]), int(args["step"])
        values = [[t, str((t - start) // step)] for t in range(start, end + 1, step)]
        result = [{"metric": {"__name__": "queue_depth", "worker": w}, "values": values} for w in ("prefill", "decode")]
        self.send_response(200); self.end_headers()
        self.wfile.write(json.dumps({"status": "success", "data": {"resultType": "matrix", "result": result}}).encode())

server = ThreadingHTTPServer(("127.0.0.1", 0), StandIn)
threading.Thread(target=server.serve_forever, daemon=True).start()
prom = PrometheusConnect(url=f"http://127.0.0.1:{server.server_address[1]}")
end = datetime(2026, 1, 1, tzinfo=timezone.utc)
frame = MetricRangeDataFrame(prom.custom_query_range("queue_depth", start_time=end - timedelta(minutes=10), end_time=end, step="60"))
server.shutdown()
assert len(frame) == 22 and sorted(frame["worker"].unique()) == ["decode", "prefill"]
assert frame[frame["worker"] == "decode"]["value"].tolist() == [float(i) for i in range(11)]
print(frame.groupby("worker")["value"].agg(["count", "max"]))

## Example 3: `Metric` objects, merging two answers for the same series

In [ ]:
import datetime

from prometheus_api_client import Metric, MetricsList

def series(job, start, count, value):
    return {"metric": {"__name__": "up", "job": job}, "values": [[1700000000 + 60 * i, value] for i in range(start, start + count)]}

metrics = MetricsList([series("planner", 0, 5, "1"), series("frontend", 0, 5, "1"), series("planner", 5, 5, "0")])
assert len(metrics) == 2, "two answers for the planner series are merged into one Metric"
planner = next(m for m in metrics if m.label_config == {"job": "planner"})
assert planner.metric_name == "up" and planner.metric_values["y"].tolist() == [1.0] * 5 + [0.0] * 5
recent = Metric(series("planner", 0, 5, "1"), oldest_data_datetime=datetime.timedelta(minutes=3))
merged = recent + Metric(series("planner", 5, 5, "0"))
assert len(merged.metric_values) == 4, "only the newest 3 minutes (4 samples) are kept"
try:
    planner + Metric(series("frontend", 0, 1, "1"))
    raise AssertionError("adding two different series must fail")
except TypeError as error:
    assert "Different metric labels" in str(error)
print("planner up/down samples:", planner.metric_values["y"].tolist())

## Example 4: an instant answer as a `MetricSnapshotDataFrame`

In [ ]:
from prometheus_api_client import MetricSnapshotDataFrame
from prometheus_api_client.exceptions import MetricValueConversionError

answer = [
    {"metric": {"__name__": "gpu_util", "gpu": "0", "node": "a"}, "value": [1700000000, "0.91"]},
    {"metric": {"__name__": "gpu_util", "gpu": "1", "node": "a"}, "value": [1700000000, "0.35"]},
    {"metric": {"__name__": "gpu_util", "gpu": "0", "node": "b"}, "value": [1700000060, "0.77"]},
]
frame = MetricSnapshotDataFrame(answer)
assert list(frame.columns) == ["__name__", "gpu", "node", "timestamp", "value"], list(frame.columns)
frame["value"] = frame["value"].astype(float)
busiest = frame.sort_values("value", ascending=False).iloc[0]
assert (busiest["node"], busiest["gpu"], busiest["value"]) == ("a", "0", 0.91)
assert str(frame["timestamp"].max()) == "2023-11-14 22:14:20"
try:
    from prometheus_api_client import Metric
    Metric({"metric": {"__name__": "bad"}, "value": [1700000000, "not-a-number"]})
    raise AssertionError("a non-numeric sample must be refused")
except MetricValueConversionError:
    pass
print(frame.groupby("node")["value"].mean())

## Example 5: errors from the server, and the time helpers

In [ ]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse
from datetime import timedelta
from prometheus_api_client import PrometheusApiClientException, PrometheusConnect
from prometheus_api_client.utils import parse_datetime, parse_timedelta

class Refusing(BaseHTTPRequestHandler):  # a server that refuses the query, as Prometheus does a malformed one
    def log_message(self, *args): pass
    def do_GET(self):
        self.send_response(400); self.end_headers()
        self.wfile.write(json.dumps({"status": "error", "errorType": "bad_data", "error": "parse error"}).encode())

server = ThreadingHTTPServer(("127.0.0.1", 0), Refusing)
threading.Thread(target=server.serve_forever, daemon=True).start()
try:
    PrometheusConnect(url=f"http://127.0.0.1:{server.server_address[1]}").custom_query("sum(")
    raise AssertionError("a 400 answer must raise")
except PrometheusApiClientException as error:
    assert "400" in str(error) and "parse error" in str(error)
finally:
    server.shutdown()
assert parse_datetime("2026-01-02 03:04:05").isoformat() == "2026-01-02T03:04:05"
assert abs(parse_timedelta("now", "2h") - timedelta(hours=2)) < timedelta(seconds=5)
print("400 raised PrometheusApiClientException; parse_timedelta('now', '2h') =", parse_timedelta("now", "2h"))